# Raytracing ray image

The `batcamp` library also provides a ray tracer supporting `rpa` and `xyz` octrees. 

In [ ]:
from batread import Dataset
import matplotlib.pyplot as plt
import numpy as np

from batcamp import Octree, OctreeInterpolator, OctreeRayTracer

ds = Dataset.from_file("../sample_data/3d__var_2_n00060005.plt")
print(ds)


tree = Octree.from_ds(ds)
print(tree)

interp = OctreeInterpolator(tree, ds["Rho [g/cm^3]"])
print(interp)

tracer = OctreeRayTracer(tree)
print(tracer)

In [ ]:
coords = np.linspace(-215.0, 215.0, 512)
y, z = np.meshgrid(coords, coords, indexing="xy")
origins = np.stack((np.full_like(y, -400.0), y, z), axis=-1)
directions = np.array([1.0, 0.0, 0.0], dtype=float)

origins.shape, directions.shape


In [ ]:
segments = tracer.trace(origins, directions)
cell_counts = np.diff(segments.ray_offsets).reshape(y.shape)

int(cell_counts.min()), float(np.median(cell_counts)), int(cell_counts.max())


In [ ]:
rho_los, _ = tracer.trilinear_image(interp, origins, directions)

fig, ax = plt.subplots(figsize=(6, 5))
mesh = ax.pcolormesh(y, z, rho_los, shading="auto", norm="log")
ax.set_xlabel("y")
ax.set_ylabel("z")
ax.set_title("Cartesian ray-integrated density")
fig.colorbar(mesh, ax=ax, label="line integral")
plt.show()

rho_los.shape
